In [14]:
# Load the important libraries
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from photutils.centroids import centroid_sources, centroid_com
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.visualization import simple_norm
stretch_ratio = 95.0 # for visualization using simple_norm
cmap = 'gnuplot'

In [2]:
# Define a function to perforrm coordinate transformation
def transform_pixels(x1, y1, wcs1, wcs2):
    """ A function to transform the (x,y) pixels from one WCS to another.
    Parameters
    ----------
    x1: float or array_like
        x-coordinate of the pixel from data with wcs1
    y1: float or array_like 
        y-coordinate of the pixel from data with wcs1
    wcs1: astropy WCS object
        The world coordinate system of the first data set.
    wcs2: astropy WCS object
        The world coordinate system of the second data set.
        
    Returns
    -------
    x2: float or array_like
        The corresponding x-coordinate from the data set with wcs2
    y2: float or array_like
        The corresponding y-coordinate from the data set with wcs2
    
    """
    r, d = wcs1.pixel_to_world_values(x1, y1)
    x2, y2 = wcs2.world_to_pixel_values(r, d)
    return x2, y2


In [3]:
!pwd

/home/ahmed/kcwi_reduction/redux


In [4]:
!ls ../*txt

../allb.txt
../arcs2x2MedKBlueBLFeAr4500_10.0_5dea.txt
../arcs2x2MedKBlueBLThAr4500_20.0_5dea.txt
../bias2x2TUP010_5dea.txt
../cbars2x2MedKBlueBL_4500_0.7_5dea.txt
../cflat2x2MedKBlueBL_4500_0.7_5dea.txt
../dflat2x2MedKBlueBL_4500_14.0_5dea.txt
../g191b2b2x2MedKBlueBL4500_5dea.txt
../J045313-1305552x2MedKBlueBL4500_5dea.txt


In [5]:
!cat ../J045313-1305552x2MedKBlueBL4500_5dea.txt

kb180215_00075.fits
kb180215_00076.fits
kb180215_00077.fits
kb180215_00078.fits
kb180215_00079.fits
kb180215_00080.fits


In [6]:
!pwd

/home/ahmed/kcwi_reduction/redux


In [7]:
file_directory = '/home/ahmed/kcwi_reduction/redux/'


infils = ['kb180215_000'+str(i)+'_icubes.fits' for i in range(75,81)]#range(58, 61)]

verbose = 0
infils

['kb180215_00075_icubes.fits',
 'kb180215_00076_icubes.fits',
 'kb180215_00077_icubes.fits',
 'kb180215_00078_icubes.fits',
 'kb180215_00079_icubes.fits',
 'kb180215_00080_icubes.fits']

In [15]:
f0 = fits.open(file_directory + infils[0])

cmap = 'inferno'#'viridis'
max_factor = 0.3


# Loading the science info from index 0 of the KCWI fits file
hdr_0 = f0[0].header  # header of the primary HDU (Flux cube)
data_0 = f0[0].data   # data of the primary HDU
wcs_0 = WCS(hdr_0).celestial

# create white light image by summing along the wavelength axis
wl_0 = np.nanmedian(data_0[100:-100,:,:], axis=0)

norm_0 = simple_norm(wl_0, stretch='linear', percent=stretch_ratio)

#%matplotlib inline
%matplotlib qt
plt.close()
fig = plt.figure(1, figsize=(4,4), dpi=300)
ax1 = fig.add_subplot(111)
im1 = ax1.imshow(wl_0, origin='lower', interpolation='nearest', cmap=cmap, norm=norm_0)
fig.colorbar(im1, ax=ax1)
fig.tight_layout()
plt.show(block=False)
# Get input through mouse clicks instead
print("Click on the plot to select points, then close the plot or press Enter")
points = plt.ginput(n=-1, timeout=0)  # n=-1 means unlimited points
print(f"You selected points: {points}")

x0_guess = [points[l][0] for l in range(len(points))]
y0_guess = [points[l][1] for l in range(len(points))]


Set MJD-BEG to 58164.236661 from DATE-BEG.
Set MJD-END to 58164.250550 from DATE-END'. [astropy.wcs.wcs]


Click on the plot to select points, then close the plot or press Enter
You selected points: [(12.740526231349278, 23.993021017166697)]


In [16]:
# selecting a box size of 3 arcseconds for both instruments
box_xside_0 = 3#int(1.7 / (max(np.abs([hdr_0["CD1_1"], hdr_0["CD1_2"]])) * 3600.))
box_yside_0 = 3#int(1.7 / (max(np.abs([hdr_0["CD2_2"], hdr_0["CD2_1"]])) * 3600.))

if box_xside_0 % 2 == 0:
        box_xside_0 += 1
if box_yside_0 % 2 == 0:
        box_yside_0 += 1
    

box_size_0 = (box_yside_0, box_xside_0)
box_size_0

(3, 3)

In [17]:

# calculating the Center of Mass (x, y) coordinates of the points sources in the MUSE white light image    
x_com_0, y_com_0 = centroid_sources(wl_0, x0_guess, y0_guess, 
                                    box_size=box_size_0, centroid_func=centroid_com)

# Convert from (x_com, y_com) to RA and DEC
ra_0, dec_0 = wcs_0.pixel_to_world_values(x_com_0, y_com_0)
c_0 = SkyCoord(ra_0, dec_0, frame='icrs', unit='deg')

In [18]:
verbose = False
SAVE = 1#False
c_mark = 'g'   # color of the point sources markers


#%matplotlib qt
%matplotlib qt
for i in range(1, len(infils)):
    f = fits.open(file_directory + infils[i])
    data_f = f[0].data
    hdr_f = f[0].header
    wcs_f = WCS(hdr_f).celestial

    wl_f = np.nanmedian(data_f[100:-100,:,:], axis=0)

    norm_f = simple_norm(wl_f, stretch='linear', percent=stretch_ratio) 

    
    #plt.close()
    fig = plt.figure(i+100, figsize=(3,6), dpi=300)
    ax1 = fig.add_subplot(111)
    ax1.imshow(wl_f, origin='lower', interpolation='nearest', cmap=cmap, norm=norm_f)
    ax1.set_title("Select Points for Frame #"+str(i)+", Click Enter")
    
    fig.tight_layout()
    plt.show(block=False)
    # Get input through mouse clicks instead
    print("Click on the plot to select points, then close the plot or press Enter")
    points = plt.ginput(n=-1, timeout=0)  # n=-1 means unlimited points
    print(f"You selected points: {points}")
    #plt.draw()
    #plt.pause(0.01)

    xf_guess = [points[l][0] for l in range(len(points))]
    yf_guess = [points[l][1] for l in range(len(points))]

    # selecting a box size of 3 arcseconds for both instruments
    box_xside_f = 5#int(2.0 / (max(np.abs([hdr_f["CD1_1"], hdr_f["CD1_2"]])) * 3600.))
    box_yside_f = 5#int(2.0 / (max(np.abs([hdr_f["CD2_2"], hdr_f["CD2_1"]])) * 3600.))

    
    if box_xside_f % 2 == 0:
            box_xside_f += 1
    if box_yside_f % 2 == 0:
            box_yside_f += 1
    
    box_size_f = (box_yside_f, box_xside_f)
    # calculating the Center of Mass (x, y) coordinates of the points sources in the MUSE white light image    
    x_com_f, y_com_f = centroid_sources(wl_f, xf_guess, yf_guess, 
                                        box_size=box_size_f, centroid_func=centroid_com)
    if verbose == True:
        # Make a figure to show the two images together with the point-sources marked in both images
        mark = 3
        fig = plt.figure(2, figsize=(5,6), dpi=300)
        ax1 = fig.add_subplot(121)
        ax1.imshow(wl_0, origin='lower', interpolation='nearest', cmap=cmap, norm=norm_0)
        ax1.set_title("Reference Frame")#+hdr_h["PHOTMODE"])  # This line may change depending on the header structure of the reference image
        #ax1.set_xlim([450, 650])
        #ax1.set_ylim([550, 700])
        ax1.plot(x_com_0, y_com_0, 's', c=c_mark, markersize=mark)
    
        ax2 = fig.add_subplot(122)
        ax2.imshow(wl_f, origin='lower', interpolation='nearest', cmap=cmap, norm=norm_f)
        ax2.set_title("Frame #"+str(i))
        ax2.plot(x_com_f, y_com_f, 's', c=c_mark, markersize=mark)
    
        fig.tight_layout()
        plt.show()
        plt.show(block=False)

    # Convert from (x_com, y_com) to RA and DEC
    ra_f, dec_f = wcs_f.pixel_to_world_values(x_com_f, y_com_f)

    c_f = SkyCoord(ra_f, dec_f, frame='icrs', unit='deg')

    offsets = c_f.separation(c_0)
    print("SkyCoord Offsets in arcseconds: ", offsets.arcsecond)
    print("SkyCoord Offsets in degrees: ", offsets.degree)


    
    ra_offset = (c_0.ra.degree - c_f.ra.degree) * np.cos(c_f.dec.radian)
    dec_offset = (c_0.dec.degree - c_f.dec.degree)
    # Calculating the difference between the RA, DEC of the point sources in both images
    delta_ra = ra_offset                        # point sources offset in RA (in degrees)
    delta_dec = dec_offset                      # point sources offset in Decl. (in degress)
    
    average_delta_ra =  np.average(delta_ra)   # the mean offset in RA of all point sources (in degrees) 
    average_delta_dec = np.average(delta_dec)  # the mean offset in Decl. of all point sourcs (in degrees)

    print("RA Offset: ", delta_ra * 3600., "arcseconds")
    print("Decl. Offset: ", delta_dec * 3600., "arcseconds") 
    print("average RA offset: ", np.average(delta_ra)*3600., "arcseconds")
    print("average Decl. offset: ", np.average(delta_dec)*3600., "arcseconds")

    # Update the cube headers
    from copy import deepcopy

    # copying the flux data-cube header and modifying it by applying the offset for the (RA, Decl.)
    hdr_f_new = deepcopy(hdr_f)
    hdr_f_new["CRVAL1"] = hdr_f["CRVAL1"] + average_delta_ra
    hdr_f_new["CRVAL2"] = hdr_f["CRVAL2"] + average_delta_dec

    # Check the results visually
    wcs_f_new = WCS(hdr_f_new).celestial
    
    x_old, y_old = transform_pixels(x_com_0, y_com_0, wcs_0, wcs_f)     # pixels of the point sources using the old header of the data cube
    x_new, y_new = transform_pixels(x_com_0, y_com_0, wcs_0, wcs_f_new) # pixels of the point sources using the new header of the data cube

    #%matplotlib inline
    #%matplotlib qt
    mark = 3
    fig = plt.figure(200+i, figsize=(4,6), dpi=300)
    ax1 = fig.add_subplot(121)
    ax1.imshow(wl_0, origin='lower', interpolation='nearest', cmap=cmap, vmin=-0.001, vmax=max_factor*wl_0.max())
    ax1.set_title("Reference Frame")#+hdr_h["PHOTMODE"]) # This line may change depending on the structure of the fits file of the reference image
    ax1.plot(x_com_0, y_com_0, 's', c=c_mark, markersize=mark)

    ax2 = fig.add_subplot(122)
    ax2.imshow(wl_f, origin='lower', interpolation='nearest', cmap=cmap, vmin=-0.001, vmax=max_factor*wl_f.max())
    ax2.set_title("Frame #"+str(i))
    ax2.plot(x_old, y_old, 's', c=c_mark, markersize=mark, label="Old")
    ax2.plot(x_new, y_new, 'D', c='c', markersize=mark, label="New")
    ax2.legend(loc=0)

    fig.tight_layout()
    plt.show()
    plt.show(block=False)

    f[0].header = hdr_f_new
    # Save the file
    if SAVE == True:
        f.writeto(file_directory+infils[i], overwrite=True)
        print("==========================================")
        print("SAVED Updated File: "+file_directory+infils[i])
        print("==========================================")
    f.close()
    

Set MJD-BEG to 58164.251221 from DATE-BEG.
Set MJD-END to 58164.265110 from DATE-END'. [astropy.wcs.wcs]


Click on the plot to select points, then close the plot or press Enter
You selected points: [(12.702009518773137, 24.508461131676356)]
SkyCoord Offsets in arcseconds:  [0.24532888]
SkyCoord Offsets in degrees:  [6.81469101e-05]
RA Offset:  [-0.22517785] arcseconds
Decl. Offset:  [-0.09737145] arcseconds
average RA offset:  -0.22517784746929898 arcseconds
average Decl. offset:  -0.09737145439387973 arcseconds
SAVED Updated File: /home/ahmed/kcwi_reduction/redux/kb180215_00076_icubes.fits


Set MJD-BEG to 58164.251221 from DATE-BEG.
Set MJD-END to 58164.265110 from DATE-END'. [astropy.wcs.wcs]
Set MJD-BEG to 58164.265783 from DATE-BEG.
Set MJD-END to 58164.279672 from DATE-END'. [astropy.wcs.wcs]


Click on the plot to select points, then close the plot or press Enter
You selected points: [(12.159439450026442, 24.44817556848228)]
SkyCoord Offsets in arcseconds:  [0.25586385]
SkyCoord Offsets in degrees:  [7.10732906e-05]
RA Offset:  [-0.19222256] arcseconds
Decl. Offset:  [0.16886914] arcseconds
average RA offset:  -0.19222256054866121 arcseconds
average Decl. offset:  0.16886914421050392 arcseconds
SAVED Updated File: /home/ahmed/kcwi_reduction/redux/kb180215_00077_icubes.fits


Set MJD-BEG to 58164.265783 from DATE-BEG.
Set MJD-END to 58164.279672 from DATE-END'. [astropy.wcs.wcs]
Set MJD-BEG to 58164.281066 from DATE-BEG.
Set MJD-END to 58164.294955 from DATE-END'. [astropy.wcs.wcs]


...

Set MJD-BEG to 58164.281066 from DATE-BEG.
Set MJD-END to 58164.294955 from DATE-END'. [astropy.wcs.wcs]
Set MJD-BEG to 58164.295628 from DATE-BEG.
Set MJD-END to 58164.309517 from DATE-END'. [astropy.wcs.wcs]


Click on the plot to select points, then close the plot or press Enter
You selected points: [(20.659703860391332, 66.1657852987837)]
SkyCoord Offsets in arcseconds:  [0.52441367]
SkyCoord Offsets in degrees:  [0.00014567]
RA Offset:  [-0.42182945] arcseconds
Decl. Offset:  [0.31155987] arcseconds
average RA offset:  -0.421829445223298 arcseconds
average Decl. offset:  0.31155986561230975 arcseconds
SAVED Updated File: /home/ahmed/kcwi_reduction/redux/kb180215_00079_icubes.fits


Set MJD-BEG to 58164.295628 from DATE-BEG.
Set MJD-END to 58164.309517 from DATE-END'. [astropy.wcs.wcs]
Set MJD-BEG to 58164.310189 from DATE-BEG.
Set MJD-END to 58164.324078 from DATE-END'. [astropy.wcs.wcs]


Click on the plot to select points, then close the plot or press Enter
You selected points: [(20.719989423585403, 66.76864093072447)]
SkyCoord Offsets in arcseconds:  [0.65730028]
SkyCoord Offsets in degrees:  [0.00018258]
RA Offset:  [-0.6307542] arcseconds
Decl. Offset:  [0.18491272] arcseconds
average RA offset:  -0.6307541991049386 arcseconds
average Decl. offset:  0.18491272419183247 arcseconds
SAVED Updated File: /home/ahmed/kcwi_reduction/redux/kb180215_00080_icubes.fits


Set MJD-BEG to 58164.310189 from DATE-BEG.
Set MJD-END to 58164.324078 from DATE-END'. [astropy.wcs.wcs]
